# Fase 2 — Detecção de EPIs

Fine-tuning do YOLOv8n para as 10 classes de EPI, a partir dos dados preparados no notebook `01_data.ipynb` (precisa ter sido executado antes, ou os dados já estarem no mesmo `PROJECT_DIR` no Google Drive).

## Setup

In [ ]:
!pip install -q ultralytics opencv-python pandas pyarrow matplotlib pillow
# --upgrade e necessario: o Colab ja vem com uma versao antiga (2.0.2) do
# pacote kaggle pre-instalada, que nao suporta o token novo nem "python -m kaggle".
!pip install -q --upgrade kaggle

from pathlib import Path

# Armazenamento persistente compartilhado entre os notebooks: monta o Google
# Drive e usa uma pasta fixa. Troque o caminho se preferir outra estrutura.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/vc-seguranca-trabalho')
except ImportError:
    # Execucao fora do Colab (teste local) - usa uma pasta local.
    PROJECT_DIR = Path('./vc-seguranca-trabalho').resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ROOT = PROJECT_DIR
print("Diretorio do projeto:", ROOT)


## 5. Preparar configuração YOLO (detecção)

Gera `data.yaml` e as listas `train.txt`/`val.txt`/`test.txt` no formato Ultralytics, a partir dos splits do notebook de dados.

In [ ]:
from pathlib import Path

DATASET_DIR = ROOT / "data" / "raw" / "construction-site-safety"
SPLITS_DIR = ROOT / "data" / "splits" / "construction-site-safety"

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]

for split in ["train", "val", "test"]:
    manifest = SPLITS_DIR / f"{split}.txt"
    filenames = [
        line.strip() for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()
    ]
    out_path = DATASET_DIR / f"{split}.txt"
    images_dir = DATASET_DIR / "images"
    with open(out_path, "w", encoding="utf-8") as f:
        for name in filenames:
            f.write((images_dir / name).as_posix() + "\n")
    print(f"{split}: {len(filenames)} imagens -> {out_path}")

data_yaml = DATASET_DIR / "data.yaml"
with open(data_yaml, "w", encoding="utf-8") as f:
    f.write(f"path: {DATASET_DIR.as_posix()}\n")
    f.write("train: train.txt\n")
    f.write("val: val.txt\n")
    f.write("test: test.txt\n")
    f.write(f"nc: {len(CLASS_NAMES)}\n")
    f.write(f"names: {CLASS_NAMES}\n")

print(f"data.yaml (Ultralytics) escrito em: {data_yaml}")


## 6. Treinar o detector (YOLOv8n)

In [ ]:
from ultralytics import YOLO

DATA_YAML = ROOT / "data" / "raw" / "construction-site-safety" / "data.yaml"
PROJECT_DIR = ROOT / "models" / "detection"
RUN_NAME = "css_yolov8n_baseline"

# Hiperparametros documentados em docs/relatorio-tecnico.md (secao 3.1).
# Com GPU (Colab), este treino roda em poucos minutos - no desenvolvimento
# original (CPU), levou ~59 minutos para as mesmas 30 epocas.
HYPERPARAMS = dict(
    model="yolov8n.pt",
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer="auto",
    seed=42,
    patience=100,
)

model = YOLO(HYPERPARAMS["model"])
model.train(
    data=str(DATA_YAML),
    epochs=HYPERPARAMS["epochs"],
    imgsz=HYPERPARAMS["imgsz"],
    batch=HYPERPARAMS["batch"],
    optimizer=HYPERPARAMS["optimizer"],
    seed=HYPERPARAMS["seed"],
    patience=HYPERPARAMS["patience"],
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)
print(f"Treino concluido. Resultados em: {PROJECT_DIR / RUN_NAME}")


## Resultado esperado

mAP@0.5 ≈ 0.49 (validação) — ver `docs/relatorio-tecnico.md` seção 4.1 para os números completos obtidos no desenvolvimento original.